In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.metrics import (accuracy_score,f1_score,roc_auc_score,confusion_matrix,classification_report)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
!pip install xgboost
from xgboost import XGBClassifier


from sklearn.model_selection import cross_val_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
TW = pd.read_csv(
    r"../classification/Twitter/Absolute_labeling/Twitter-Absolute-Sigma-500.data",
    sep=",",
    header=None
)

groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label")  # остання колонка

TW.columns = columns

TW.head(2)

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,AI_0,AI_1,AI_2,...,ADL_5,ADL_6,NAD_0,NAD_1,NAD_2,NAD_3,NAD_4,NAD_5,NAD_6,label
0,889,939,960,805,805,1143,1121,549,613,587,...,1.0,1.0,889,939,960,805,805,1143,1121,1.0
1,542,473,504,626,647,795,832,366,288,318,...,1.0,1.0,542,473,504,626,647,795,832,1.0


In [3]:
TH = pd.read_csv(
    "../classification/TomsHardware/Absolute_labeling/TomsHardware-Absolute-Sigma-500.data",
    sep=",",
    header=None
)


groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = []
for group in groups:
    for t in range(8):
        columns.append(f"{group}_{t}")

columns.append("label")  # остання колонка

TH.columns = columns


prefixes = {col.split("_")[0] for col in TH.columns if "_" in col}


TH.head()

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,NCD_7,BL_0,BL_1,...,AS_NA_7,AS_NAC_0,AS_NAC_1,AS_NAC_2,AS_NAC_3,AS_NAC_4,AS_NAC_5,AS_NAC_6,AS_NAC_7,label
0,1,0,0,0,0,0,0,1,1.0,0.0,...,0.001816,0.001211,0.000560,0.000000,0.000000,0.000161,0.0,0.000301,0.000818,1.0
1,1,1,1,1,0,0,0,0,1.0,1.0,...,0.005029,0.000784,0.000802,0.001592,0.001612,0.000741,0.0,0.000545,0.002437,1.0
2,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
3,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
4,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0


### Twitter

In [15]:
X = TW.drop(columns=["label"])
y = TW["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

### Random Forest baseline

In [5]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
pred_proba = rf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_rf)
precision = precision_score(y_test, pred_rf, zero_division=0)
recall = recall_score(y_test, pred_rf, zero_division=0)
f1 = f1_score(y_test, pred_rf, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Random Forest baseline")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Random Forest baseline
Accuracy: 0.9685168076185061
Precision: 0.9295308187672493
Recall: 0.9094509450945094
F1-score: 0.9193812556869881
ROC-AUC: 0.9913626466910204


### Random Forest with class_weight

In [7]:
rf = RandomForestClassifier(random_state=42,class_weight='balanced')
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
pred_proba = rf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_rf)
precision = precision_score(y_test, pred_rf, zero_division=0)
recall = recall_score(y_test, pred_rf, zero_division=0)
f1 = f1_score(y_test, pred_rf, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Random Forest with class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Random Forest with class_weight
Accuracy: 0.968232535001066
Precision: 0.9378170204771745
Recall: 0.8986498649864987
F1-score: 0.9178157749586321
ROC-AUC: 0.9915627729594583


### Random Forest with Grid Search

In [22]:
import pandas as pd

data = {
    "Model": [
        "Random Forest",
        "Random Forest with class_weight",
        "Random Forest with RandomizedSearch"
    ],
    "Accuracy": [0.9685, 0.9682, 0.9690],
    "Precision ": [0.9295, 0.9378, 0.9338],
    "Recall ": [0.9095, 0.8986, 0.9087],
    "F1-score": [0.9194, 0.9178, 0.9211],
    "ROC-AUC": [0.9914, 0.9916, 0.9762]
}

df_rf_tw = pd.DataFrame(data)
df_rf_tw

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Random Forest,0.9685,0.9295,0.9095,0.9194,0.9914
1,Random Forest with class_weight,0.9682,0.9378,0.8986,0.9178,0.9916
2,Random Forest with RandomizedSearch,0.9690,0.9338,0.9087,0.9211,0.9762


### TOm Harware

In [25]:
X = TH.drop(columns=["label"])
y = TH["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

### Random Forest baseline

In [11]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
pred_proba = rf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_rf)
precision = precision_score(y_test, pred_rf, zero_division=0)
recall = recall_score(y_test, pred_rf, zero_division=0)
f1 = f1_score(y_test, pred_rf, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Random Forest baseline")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Random Forest baseline
Accuracy: 0.9664769133459835
Precision: 0.9761658031088083
Recall: 0.9691358024691358
F1-score: 0.9726381001548787
ROC-AUC: 0.995300262860927


### Random Forest with class_weight

In [12]:
rf = RandomForestClassifier(random_state=42,class_weight='balanced')
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
pred_proba = rf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_rf)
precision = precision_score(y_test, pred_rf, zero_division=0)
recall = recall_score(y_test, pred_rf, zero_division=0)
f1 = f1_score(y_test, pred_rf, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Random Forest with class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Random Forest with class_weight
Accuracy: 0.9702719797596457
Precision: 0.976313079299691
Recall: 0.9753086419753086
F1-score: 0.9758106021616058
ROC-AUC: 0.9950679789440964


### Random Forest with class_weight

In [26]:
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", 0.5]
}

grid = GridSearchCV(rf_base, param_grid, cv=5, scoring="f1", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

best_rf = grid.best_estimator_


Best params: {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 200}
Best CV F1: 0.9776531328254899


In [27]:
print("Random Forest with рандом")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Random Forest with рандом
Accuracy: 0.9702719797596457
Precision: 0.976313079299691
Recall: 0.9753086419753086
F1-score: 0.9758106021616058
ROC-AUC: 0.9950679789440964


In [29]:
import pandas as pd

data = {
    "Model": [
        "Random Forest",
        "Random Forest with class_weight",
        "Random Forest with RandomizedSearch"
    ],
    "Accuracy": [0.9665, 0.9703, 0.9703],
    "Precision ": [0.9762, 0.9763, 0.9763],
    "Recall ": [0.9691, 0.9753, 0.9753],
    "F1-score ": [0.9726, 0.9758, 0.9758],
    "ROC-AUC": [0.9953, 0.9951, 0.9951]
}

df_rf_th = pd.DataFrame(data)
df_rf_th

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Random Forest,0.9665,0.9762,0.9691,0.9726,0.9953
1,Random Forest with class_weight,0.9703,0.9763,0.9753,0.9758,0.9951
2,Random Forest with RandomizedSearch,0.9703,0.9763,0.9753,0.9758,0.9951
